# TFM - Predictive Modeling

### Objective

The main objective of this notebook is to develop and train supervised regression models to predict the economic value of MLB hitters using the modeling dataset prepared during the feature engineering phase.

This phase includes splitting the data into training and testing sets, establishing a baseline model, training selected linear and non-linear regression algorithms, and performing a limited hyperparameter tuning process. The trained models and their predictions will be saved for their detailed evaluation in the next phase of the project.

Notebook: 06_Predictive_Modeling

Author: Ronald Báez


In [1]:
# Import libraries

# Data manipulation
import pandas as pd
import numpy as np

# Data splitting and preprocessing
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Regression models
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

# Model persistence
import joblib

# Reproducibility
RANDOM_STATE = 42

### Load Modeling Data

The modeling dataset generated during the feature engineering phase is loaded. The predictor variables and the target variable are then separated before starting the model training process.

In [2]:
# Load the modeling dataset
df = pd.read_csv("../01_data/03_final_data/mlb_hitters_modeling.csv")

print(f"Dataset dimensions: {df.shape}")
print(f"Number of columns: {df.shape[1]}")

display(df.head())

Dataset dimensions: (3684, 13)
Number of columns: 13


,year,age,age_squared,pa,war,ops_plus,rbi,sb,hr_rate,bb_rate,so_rate,salary,log_salary
0,2017,36.0,1296.0,163.0,0.6,82.0,14.0,0.0,0.036810,0.073620,0.177914,2500000,14.731802
1,2018,37.0,1369.0,183.0,0.2,104.0,15.0,0.0,0.005464,0.142077,0.202186,1250000,14.038655
2,2016,25.0,625.0,227.0,-0.2,59.0,22.0,7.0,0.017621,0.101322,0.303965,515500,13.152895
3,2017,26.0,676.0,412.0,1.9,122.0,65.0,5.0,0.046117,0.077670,0.252427,538500,13.196545
4,2018,27.0,729.0,285.0,-0.9,69.0,38.0,3.0,0.028070,0.126316,0.319298,440336,12.995296


### Train-Test Split

The predictor variables and the target variable are separated. The dataset is then divided into training and testing sets, using 80% of the records for training and 20% for testing. A fixed random state is used to ensure reproducibility.

In [3]:
# Separate predictor variables and target
X = df.drop(columns= ["log_salary", "salary"])
y = df["log_salary"]

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE
)

print(f"Training features: {X_train.shape}")
print(f"Testing features: {X_test.shape}")
print(f"Training target: {y_train.shape}")
print(f"Testing target: {y_test.shape}")

Training features: (2947, 11)
Testing features: (737, 11)
Training target: (2947,)
Testing target: (737,)


###  Baseline Model

A baseline regression model is trained using the mean value of the target variable. This model provides a minimum reference against which the performance of the predictive models will be compared during the evaluation phase.

In [4]:
# Define and train the baseline model
baseline_model = DummyRegressor(strategy="mean")
baseline_model.fit(X_train, y_train)

# Generate baseline predictions
y_pred_baseline = baseline_model.predict(X_test)

print("Baseline model trained successfully.")
print(f"Baseline predictions generated: {len(y_pred_baseline)}")

Baseline model trained successfully.
Baseline predictions generated: 737


###  Train Selected Models

Four regression models are trained to compare linear, regularized and non-linear approaches. Standardization is applied to the linear models through pipelines, while tree-based models are trained directly on the original feature values.

In [5]:
# Define the regression models
models = {
    "Linear Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LinearRegression())
    ]),
    
    "Ridge Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", Ridge())
    ]),
    
    "Decision Tree": DecisionTreeRegressor(
        random_state=RANDOM_STATE
    ),
    
    "Random Forest": RandomForestRegressor(
        n_estimators=100,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
}

# Train the models
for name, model in models.items():
    model.fit(X_train, y_train)
    print(f"{name} trained successfully.")

Linear Regression trained successfully.
Ridge Regression trained successfully.
Decision Tree trained successfully.
Random Forest trained successfully.


###  Basic Hyperparameter Tuning

A limited grid search with five-fold cross-validation is performed for the models that require hyperparameter configuration. The search is restricted to a small number of combinations to improve model performance without introducing unnecessary computational complexity.

In [6]:
# Define limited hyperparameter grids
param_grids = {
    "Ridge Regression": {
        "model__alpha": [0.1, 1, 10, 100]
    },
    
    "Decision Tree": {
        "max_depth": [5, 10, None],
        "min_samples_split": [2, 5],
        "min_samples_leaf": [1, 2]
    },
    
    "Random Forest": {
        "n_estimators": [100, 200],
        "max_depth": [10, None],
        "min_samples_leaf": [1, 2]
    }
}

# Store the tuned models
tuned_models = {}

# Perform grid search
for name, param_grid in param_grids.items():
    grid_search = GridSearchCV(
        estimator=models[name],
        param_grid=param_grid,
        scoring="neg_root_mean_squared_error",
        cv=5,
        n_jobs=-1
    )
    
    grid_search.fit(X_train, y_train)
    tuned_models[name] = grid_search.best_estimator_
    
    print(f"{name}")
    print(f"Best parameters: {grid_search.best_params_}")
    print("-" * 50)

Ridge Regression
Best parameters: {'model__alpha': 0.1}
--------------------------------------------------
Decision Tree
Best parameters: {'max_depth': 5, 'min_samples_leaf': 2, 'min_samples_split': 2}
--------------------------------------------------
Random Forest
Best parameters: {'max_depth': 10, 'min_samples_leaf': 2, 'n_estimators': 200}
--------------------------------------------------


###  Train Final Models

The baseline model, linear regression and the best tuned versions of Ridge Regression, Decision Tree and Random Forest are trained using the complete training dataset. These models will be used to generate the final predictions.

In [7]:
# Define the final models
final_models = {
    "Baseline": baseline_model,
    "Linear Regression": models["Linear Regression"],
    "Ridge Regression": tuned_models["Ridge Regression"],
    "Decision Tree": tuned_models["Decision Tree"],
    "Random Forest": tuned_models["Random Forest"]
}

# Train the final models
for name, model in final_models.items():
    model.fit(X_train, y_train)
    print(f"{name} trained successfully.")

Baseline trained successfully.
Linear Regression trained successfully.
Ridge Regression trained successfully.
Decision Tree trained successfully.
Random Forest trained successfully.


###  Generate and Save Predictions

The final models are used to generate predictions for the test dataset. The actual and predicted values are stored for their evaluation in the next phase. The trained models are also saved for later use.

In [8]:
# Generate predictions
predictions = pd.DataFrame({
    "actual_log_salary": y_test.reset_index(drop=True),
    "baseline": final_models["Baseline"].predict(X_test),
    "linear_regression": final_models["Linear Regression"].predict(X_test),
    "ridge_regression": final_models["Ridge Regression"].predict(X_test),
    "decision_tree": final_models["Decision Tree"].predict(X_test),
    "random_forest": final_models["Random Forest"].predict(X_test)
})

# Save predictions
predictions.to_csv(
    "../01_data/03_final_data/model_predictions.csv",
    index=False
)

# Save trained models
joblib.dump(
    final_models["Baseline"],
    "../04_models/baseline.joblib"
)

joblib.dump(
    final_models["Linear Regression"],
    "../04_models/linear_regression.joblib"
)

joblib.dump(
    final_models["Ridge Regression"],
    "../04_models/ridge_regression.joblib"
)

joblib.dump(
    final_models["Decision Tree"],
    "../04_models/decision_tree.joblib"
)

joblib.dump(
    final_models["Random Forest"],
    "../04_models/random_forest.joblib"
)

print("Predictions and models saved successfully.")
print(f"Number of prediction records: {len(predictions)}")

display(predictions.head())

Predictions and models saved successfully.
Number of prediction records: 737


,actual_log_salary,baseline,linear_regression,ridge_regression,decision_tree,random_forest
0,15.096445,14.388707,14.647054,14.646410,14.079433,14.435146
1,17.216708,14.388707,16.058454,16.061364,16.060504,16.077890
2,15.796512,14.388707,14.035021,14.033500,13.570318,13.625115
3,13.265216,14.388707,13.817125,13.815806,14.286811,14.131955
4,14.220976,14.388707,14.143389,14.142807,14.136201,14.220886


###  Modeling Conclusions

In this phase, a baseline model and four supervised regression models were trained to predict the logarithm of MLB hitters' salaries. The selected approaches included linear regression, Ridge regression, a decision tree and a random forest, allowing both linear and non-linear relationships to be considered.

A limited hyperparameter tuning process was applied to Ridge regression and the tree-based models using cross-validation. The final models generated predictions for the test dataset, which were saved together with the observed values.

The detailed comparison of model performance using MAE, RMSE and R² will be carried out in the next phase.